# Iceberg migration — notebook

Paste-driven canonical notebook for the QA → prod Iceberg table + view
migration workflow. One fenced code or markdown block per cell. Cells
alternate **markdown intro → code**, so a 19-step workflow lands as 38
cells in `workflow.ipynb`.

---

### Setup

**What it does.** Bootstraps the kernel: pip-installs pandas, adds the
helpers folder to `sys.path`, imports every module used downstream, then
builds the dual-catalog SparkSession.

**Run when.** Fresh kernel. Always restart before re-running — Spark's
`getOrCreate()` returns the existing session if alive, and the tuning
configs below are silently dropped on a warm kernel.

**Gotcha.** Edit `connection`, `sparkapp`, and `CUTOFF_TS` to match your
environment. Empty `extra_conf` = local-mode reads only; populate the
commented K8s entries for cluster mode.

In [ ]:
%pip install --quiet pandas

import sys
from pathlib import Path

# Add the helpers folder to sys.path. The notebook lives next to the .py
# modules in this repo's `data migration/` folder, so the current dir is
# normally already correct; the candidate list covers running from one
# level up (the repo root) or a `data migration/` subdir.
HERE = Path.cwd()
candidates = [HERE, HERE / "data migration", HERE.parent / "data migration"]
ROOT = next((p for p in candidates if (p / "spark_session.py").is_file()), HERE)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from spark_session     import build_spark_session
from catalog_traversal import (
    list_schemas, summarize_tables, summarize_views,
    derive_partition_columns,
)
from build_datasources import build_datasources_rows, preview_rows
from template_builder  import build_template
from bundle_writer     import write_session_bundle, show_bundle_links
from session_state     import (
    load_state, save_state, upsert_row, is_pending, utc_now_iso,
)
from view_workflow     import migrate_views, validate_views, status_label
from table_properties  import (
    properties_via_sql, properties_via_iceberg,
    compare_properties, diff_qa_vs_prod,
    plan_property_sync, apply_property_plan, sync_batch, RESERVED_KEYS,
)

# ---------------------------------------------------------------------------
# Cutoff: single edit point for table migration filter (Cell 8), table
# validation (Cell 12), and view validation (Cell 20). Downstream cells
# reference these — edit only here.
# ---------------------------------------------------------------------------
CUTOFF_TS = "2026-05-12 23:59:59.999"
CUTOFF    = f"updated_at_ts<='{CUTOFF_TS}'"

# ---------------------------------------------------------------------------
# Connection details
# ---------------------------------------------------------------------------
connection = {
    "qa_hms_uri":      "thrift://172.27.5.25:9083",
    "prod_hms_uri":    "thrift://hivemetastore.prod.svc.cluster.local:9083",
    "qa_warehouse":    "abfs://tp-qa-datalake@azpcinpneupraist02.dfs.core.windows.net/qa/iceberg/",
    "prod_warehouse":  "abfs://tp-prod-datalake@azpcineupraist01.dfs.core.windows.net/prod/iceberg/",
    "default_fs":      "abfs://tp-prod-datalake@azpcineupraist01.dfs.core.windows.net/prod",
    "event_log_dir":   "abfs://tp-prod-logs@azpcineupraist01.dfs.core.windows.net/spark-history/",
    "azure_tenant":    "638fcbaf-ba4c-43e1-adae-5475c970fe10",
    "azure_client_id": "7b01d4d3-f76e-4215-b9cb-3f6814c7328d",
}

sparkapp = {
    "driver_cores":          "4",
    "driver_memory":         "20g",
    "executor_cores":        "3",
    "executor_memory":       "30g",
    "executor_instances":    "30",
    "executor_core_request": "1000m",
    "oidc_url": "https://qa.sds.ontp.app/realms/prod/protocol/openid-connect/token",
}

# K8s wiring for the notebook session — leave empty for local-mode reads.
extra_conf: dict[str, str] = {
    # "spark.master": "k8s://https://kubernetes.default.svc",
    # "spark.kubernetes.namespace": "<your-ns>",
    # "spark.kubernetes.authenticate.driver.serviceAccountName": "sds-jupyterhub-sa",
    # "spark.kubernetes.container.image": "<image>",
    # "spark.kubernetes.executor.podTemplateFile": "<abfs://...yaml>",
}

spark = build_spark_session(
    connection=connection,
    sparkapp=sparkapp,
    app_name="iceberg-migration-workflow",
    extra_conf=extra_conf or None,
)
print(f"Spark version : {spark.version}")
print(f"App ID        : {spark.sparkContext.applicationId}")
print(f"Master        : {spark.sparkContext.master}")
print(f"CUTOFF_TS     : {CUTOFF_TS}")
spark

### Discovery — list QA schemas

**What it does.** Prints every top-level namespace in the QA Iceberg
catalog (`iceberg_catalog1`).

**Run when.** You want to confirm the schema name(s) you'll migrate
from. Read-only — safe to re-run anytime.

**Gotcha.** Schema visibility depends on HMS permissions; if a schema
you expect is missing, check with platform.

In [ ]:
qa_schemas = list_schemas(spark, "iceberg_catalog1")
print(f"{len(qa_schemas)} QA schema(s):")
for s in qa_schemas:
    print(f"  - {s}")

### Discovery — table summary + per-table COUNT(*)

**What it does.** Lists every table under `selected_schemas`, derives
the partition spec (Iceberg internal → migrate.py format), and runs a
`COUNT(*)` per table (optionally filtered by `where_sql`).

**Run when.** Cell 4 has shown you the namespaces. Edit
`selected_schemas` to whichever ones you want to inspect.

**Gotcha.** Unfiltered counts are free (Iceberg manifest stats); a
filtered count may scan partitions and be slow on huge tables. Set
`where_sql = CUTOFF` (from Cell 2) for cutoff-bounded counts.

In [ ]:
selected_schemas = [
    "kg_olap_v3",
    "kg_publish_final",
    # "kg_fragment",
    # "kg_mini",
]

# Optional filter applied to the COUNT(*) query. Leave empty for unfiltered,
# or set to CUTOFF (from Cell 2) for cutoff-bounded counts.
where_sql = ""

tables_df = summarize_tables(spark, "iceberg_catalog1", schemas=selected_schemas)
print(f"{len(tables_df)} table(s) across {len(selected_schemas)} schema(s)")

# Add a qa_count column. Iceberg answers unfiltered COUNT(*) from manifest
# stats (fast); a filtered count may scan partitions.
counts, errors = [], []
for _, r in tables_df.iterrows():
    fqn = f"iceberg_catalog1.{r['schema']}.{r['table']}"
    q = f"SELECT COUNT(*) AS c FROM {fqn}"
    if where_sql:
        q += f" WHERE {where_sql}"
    try:
        counts.append(int(spark.sql(q).collect()[0]["c"]))
        errors.append("")
    except Exception as e:
        counts.append(None)
        errors.append(str(e)[:120])
tables_df = tables_df.copy()
tables_df["qa_count"]    = counts
tables_df["count_error"] = errors

with pd.option_context("display.max_rows", None, "display.max_colwidth", 120, "display.width", 240):
    display(tables_df)

### Define migration selections + build datasources rows

**What it does.** Declares the migration plan (28 example rows: 14 in
`kg_olap_v3 → kg_olap`, 14 in `kg_publish_final → kg_publish`), then
normalises into the `datasources.json` shape consumed by the ops-VM
driver and previews the result.

**Run when.** You've reviewed Cell 6's table summary and know exactly
which tables to migrate.

**Gotcha.** `k8s_name` must be unique per row (RFC-1123). Cross-batch
collisions — same bare table name in two source schemas going to two
different target schemas — require explicit suffixes (here `-olap` and
`-pub`). Partition columns are auto-derived via Py4J when omitted.

In [ ]:
selections = [
    # ----- kg_olap_v3 -> kg_olap (14 tables, suffix -olap) -----
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__rel__finding_associated_with_host__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemrelfindingassociatedwithhostpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__rel__finding_associated_with_person__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemrelfindingassociatedwithpersonpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__rel__vulnerability_finding_on_host__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseirelvulnerabilityfindingonhostpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__vulnerability__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseivulnerabilitypublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__rel__finding_associated_with_vulnerability__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemrelfindingassociatedwithvulnerabilitypublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__person__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseipersonpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__identity__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseiidentitypublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__assessment__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemassessmentpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__rel__finding_associated_with_identity__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemrelfindingassociatedwithidentitypublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__rel__person_has_identity__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseirelpersonhasidentitypublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__rel__person_owns_host__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseirelpersonownshostpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__finding__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemfindingpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_ei__host__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdseihostpublish-olap",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_olap_v3", "src_table": "sds_em__rel__finding_associated_with_assessment__publish",
     "tgt_schema": "kg_olap", "k8s_name": "sdsemrelfindingassociatedwithassessmentpublish-olap",
     "filter_expression": CUTOFF},

    # ----- kg_publish_final -> kg_publish (14 tables, suffix -pub) -----
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__rel__person_owns_host__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseirelpersonownshostpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__assessment__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemassessmentpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__rel__finding_associated_with_person__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemrelfindingassociatedwithpersonpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__rel__person_has_identity__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseirelpersonhasidentitypublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__rel__finding_associated_with_identity__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemrelfindingassociatedwithidentitypublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__vulnerability__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseivulnerabilitypublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__person__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseipersonpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__identity__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseiidentitypublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__rel__finding_associated_with_vulnerability__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemrelfindingassociatedwithvulnerabilitypublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__host__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseihostpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__rel__finding_associated_with_assessment__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemrelfindingassociatedwithassessmentpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__rel__finding_associated_with_host__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemrelfindingassociatedwithhostpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_ei__rel__vulnerability_finding_on_host__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdseirelvulnerabilityfindingonhostpublish-pub",
     "filter_expression": CUTOFF},
    {"src_schema": "kg_publish_final", "src_table": "sds_em__finding__publish",
     "tgt_schema": "kg_publish", "k8s_name": "sdsemfindingpublish-pub",
     "filter_expression": CUTOFF},
]

datasources_rows = build_datasources_rows(
    spark, qa_catalog="iceberg_catalog1",
    selections=selections, auto_derive_partitions=True,
)
print(f"\n{len(datasources_rows)} datasources row(s) built.\n")
preview_rows(datasources_rows)

### Build template + write session bundle

**What it does.** Renders the SparkApplication CR template and writes a
self-contained `sessions/<session_name>/` folder containing
`migrate.py`, `datasources.json`, `template.json`, seeded
`table_state.csv` + `view_state.csv`, and a per-bundle `README.md`.

**Run when.** Cell 8 has produced `datasources_rows`.

**Gotcha.** `overwrite=False` raises `FileExistsError` if the bundle
folder already exists — set to True to clobber. Download the folder
from JupyterLab's file browser and run `python migrate.py` on the ops
VM.

In [ ]:
session_name = "kg_olap_publish__final"

main_application_file = (
    "abfs://tp-prod-apps@azpcineupraist01.dfs.core.windows.net/"
    "prod/sds/data-analytics/lib/latest/sds-pe-utility-jobs-3.5.5_2.13.jar"
)

template = build_template(
    connection=connection,           # from Cell 2
    sparkapp=sparkapp,               # from Cell 2
    main_application_file=main_application_file,
    namespace="prod",
)

bundle = write_session_bundle(
    session_name,
    datasources_rows=datasources_rows,   # from Cell 8
    template=template,
    overwrite=False,                     # set True to clobber an existing folder
)
print(f"Bundle: {bundle}")
for f in ("migrate.py", "datasources.json", "template.json",
          "table_state.csv", "view_state.csv", "README.md"):
    print(f"  - {bundle / f}")

show_bundle_links(bundle)

### Table validation — counts + partition specs

**What it does.** Post-migration sanity check. Loads `table_state.csv`,
for each table runs filtered + unfiltered `COUNT(*)` on prod, derives
partition spec on both sides, writes verdict back to the CSV.

**Run when.** The ops VM has finished `python migrate.py` for the
bundle's tables.

**Gotcha.** `reset_validation = True` wipes prior validation columns on
every run. Set to False to keep skip-success behaviour. `where_sql`
defaults to `CUTOFF` (Cell 2) so the validation count matches what
`migrate.py` ran. The `prod_count_total` column is the UN-filtered prod
count — a sanity check that prod isn't carrying rows beyond the cutoff.

In [ ]:
bundle = Path("sessions/kg_olap_publish__final")    # adjust if path differs

# Cutoff filter applied to BOTH sides for count comparison. "" = unfiltered.
where_sql = CUTOFF

# Wipe prior validation state so the loop re-checks everything.
# False = keep skip-success behaviour.
reset_validation = True

state_df = load_state(bundle, kind="table")

if reset_validation:
    for col in ("validation_status", "validation_at",
                "qa_count", "prod_count", "prod_count_total",
                "partition_qa", "partition_prod", "partition_match",
                "validation_error"):
        if col in state_df.columns:
            state_df[col] = ""
    save_state(state_df, bundle, kind="table")
    print(f"reset: cleared validation state on {len(state_df)} row(s)")

print(f"state : {bundle / 'table_state.csv'}")
print(f"rows  : {len(state_df)}")
print(f"where : {where_sql or '<unfiltered>'}")
print("-" * 60)

for _, r in state_df.iterrows():
    key = r["table_key"]
    qa, prod = r["qa_table"], r["prod_table"]

    if not is_pending(state_df, "table_key", key, "validation_status"):
        print(f"SKIP   {key}  (validation_status already ok)")
        continue

    qfqn = f"iceberg_catalog1.{qa}"
    pfqn = f"iceberg_catalog2.{prod}"
    fields = {"validation_at": utc_now_iso()}
    try:
        # Filtered counts (BOTH sides under the same cutoff)
        q_qa   = f"SELECT COUNT(*) c FROM {qfqn}" + (f" WHERE {where_sql}" if where_sql else "")
        q_prod = f"SELECT COUNT(*) c FROM {pfqn}" + (f" WHERE {where_sql}" if where_sql else "")
        qa_c   = int(spark.sql(q_qa).collect()[0]["c"])
        prod_c = int(spark.sql(q_prod).collect()[0]["c"])

        # Unfiltered prod total — sanity check that prod isn't carrying
        # extra rows beyond the cutoff. Free on Iceberg (manifest stats).
        prod_total = int(spark.sql(f"SELECT COUNT(*) c FROM {pfqn}").collect()[0]["c"])

        # Partition specs
        p_qa   = derive_partition_columns(spark, qfqn)
        p_prod = derive_partition_columns(spark, pfqn)
        p_match = (p_qa == p_prod)

        all_ok = (qa_c == prod_c) and p_match
        fields.update({
            "qa_count":          qa_c,
            "prod_count":        prod_c,
            "prod_count_total":  prod_total,
            "partition_qa":      p_qa,
            "partition_prod":    p_prod,
            "partition_match":   "true" if p_match else "false",
            "validation_status": "ok" if all_ok else "mismatch",
            "validation_error":  "",
        })
        label = "OK   " if all_ok else "DIFF "
        extra = ""
        if prod_total != prod_c:
            extra = f"  (prod_total={prod_total:,}, +{prod_total - prod_c:,} beyond filter)"
        print(f"{label} {key}  qa={qa_c:,}  prod={prod_c:,}  partition={'match' if p_match else 'DIFF'}{extra}")
        if not p_match:
            print(f"       qa  spec: {p_qa}")
            print(f"       prod spec: {p_prod}")
    except Exception as e:
        fields.update({"validation_status": "error", "validation_error": str(e)[:200]})
        print(f"ERR   {key}: {e}")

    state_df = upsert_row(state_df, "table_key", key, fields)
    save_state(state_df, bundle, kind="table")

# Display: validation-relevant columns first; hide migration columns from view.
DISPLAY_COLS = [
    "table_key", "qa_table", "prod_table",
    "qa_count", "prod_count", "prod_count_total",
    "partition_qa", "partition_prod", "partition_match",
    "validation_status", "validation_at", "validation_error",
]
view = state_df[[c for c in DISPLAY_COLS if c in state_df.columns]].copy()
# Big numbers in nullable Int64 so they don't render in scientific notation.
for c in ("qa_count", "prod_count", "prod_count_total"):
    if c in view.columns:
        view[c] = pd.to_numeric(view[c], errors="coerce").astype("Int64")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120, "display.width", 240):
    display(view)

print(f"\nsaved -> {bundle / 'table_state.csv'}")
print("(apply_* columns hidden from view; still in the CSV.)")

### Manual ad-hoc table validation (optional, stateless)

**What it does.** Stateless `COUNT(*)` + partition spec diff for
arbitrary `(qa, prod)` pairs. No writeback to `table_state.csv`.

**Run when.** Cross-checking a table outside the current bundle, or
sanity-checking one row.

**Gotcha.** Results only live in this cell's output — nothing is
persisted. Use Cell 12 if you want the verdict in the state CSV.

In [ ]:
# Pairs of (qa_full, prod_full). Edit freely; no state is written.
pairs = [
    ("kg_publish_final.sds_em__finding__publish", "kg_publish.sds_em__finding__publish"),
    # ("kg.sds_ei__host__bms_client_extract__pid__srdm_inv", "kg.sds_ei__host__bms_client_extract__pid__srdm_inv"),
]

where_sql = ""   # "" = unfiltered, or e.g. CUTOFF (from Cell 2)

rows = []
for qa, prod in pairs:
    qfqn = f"iceberg_catalog1.{qa}"
    pfqn = f"iceberg_catalog2.{prod}"
    row = {"qa_table": qa, "prod_table": prod,
           "qa_count": None, "prod_count": None, "count_match": None,
           "partition_qa": None, "partition_prod": None, "partition_match": None,
           "error": ""}
    try:
        q_qa   = f"SELECT COUNT(*) c FROM {qfqn}" + (f" WHERE {where_sql}" if where_sql else "")
        q_prod = f"SELECT COUNT(*) c FROM {pfqn}" + (f" WHERE {where_sql}" if where_sql else "")
        row["qa_count"]    = int(spark.sql(q_qa).collect()[0]["c"])
        row["prod_count"]  = int(spark.sql(q_prod).collect()[0]["c"])
        row["count_match"] = row["qa_count"] == row["prod_count"]
        row["partition_qa"]    = derive_partition_columns(spark, qfqn)
        row["partition_prod"]  = derive_partition_columns(spark, pfqn)
        row["partition_match"] = row["partition_qa"] == row["partition_prod"]
    except Exception as e:
        row["error"] = str(e)[:200]
    print(f"{qa}  vs  {prod}  ->  count={row['count_match']}  partition={row['partition_match']}  {row['error'] or ''}")
    rows.append(row)

df = pd.DataFrame(rows)
# Nullable Int64 so big counts don't render in scientific notation.
for c in ("qa_count", "prod_count"):
    df[c] = df[c].astype("Int64")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120, "display.width", 240):
    display(df)

### Define view specs + persist to bundle

**What it does.** Declares which views to migrate and writes
`view_specs.json` next to the table-state CSV in the same bundle.

**Run when.** Tables have migrated successfully and you want to layer
views on top.

**Gotcha.** Only `view` is required per spec. Optional overrides
(`target_db`, `base_table`, `target_location`, `filter_expression`,
`join_key`, `cast_to_string`) default via
`view_workflow._resolve_spec` — override only when the default is
wrong for that view.

In [ ]:
import json

bundle = Path("sessions/kg_olap_publish__final")     # adjust if path differs

view_specs = [
    {"view": "kg.sds_ei__host__bitsight__company_assets__asset_id"},
    # add more here:
    # {"view": "kg.sds_ei__identity__ms_azure_ad_users__user_principal_name"},
    # {"view": "kg.sds_ei__host__bms_client_extract__pid"},
]

(bundle / "view_specs.json").write_text(json.dumps(view_specs, indent=2) + "\n")
print(f"{len(view_specs)} view spec(s) -> {bundle / 'view_specs.json'}")

### Pilot single-view migrate (dry-run → optional apply)

**What it does.** Picks one spec, prints the rewritten DDL via
`dry_run=True`, optionally applies on second run.

**Run when.** Piloting view migration before running the full batch.
Always run with `apply = False` first to read the DDL.

**Gotcha.** When `apply = True` it writes one row to `view_state.csv`.
The batch cell (Cell 22) will then skip that view on its next run
(skip-success). Edit the cell or pass `skip_if_ok=False` to redo.

In [ ]:
# Pick one spec to pilot (first match by substring; tweak the predicate).
pilot = next(s for s in view_specs if "bitsight" in s["view"])
print(f"Pilot view: {pilot['view']}\n")

# Step 1: dry-run — print the rewritten DDL, do NOT execute, no state writes.
print("=== DRY-RUN ===")
migrate_views(spark, bundle, [pilot], dry_run=True, skip_if_ok=False, verbose=True)

# Step 2: flip apply=True once the DDL looks right, then re-run this cell.
# Applies the pilot only; view_state.csv gets one row.
apply = False
if apply:
    print("\n=== APPLY ===")
    migrate_views(spark, bundle, [pilot], skip_if_ok=False)
else:
    print("\nApply skipped. Set `apply = True` and re-run to execute.")

### Manual single-view row-level diff (optional, stateless)

**What it does.** Full-outer joins QA vs prod for one view on
`JOIN_KEY`, shows mismatched rows with which columns differ, and emits
per-column mismatch counts. No state writeback.

**Run when.** Debugging WHY a view doesn't match (after Cell 24 has
flagged it as `mismatch`).

**Gotcha.** Slow on large views. The cell hardcodes `db`/`src_view`/
`filter_ts` — edit them inline. `CAST_TO_STRING` is the set of columns
to cast to string before comparing (works around precision differences
on long-precision attribute fields).

In [ ]:
from pyspark.sql import functions as F

# Pick a view to inspect. Defaults match the Cell 18 pilot.
db        = "kg"
src_view  = "sds_ei__host__bitsight__company_assets__asset_id"
tgt_view  = src_view
filter_ts = CUTOFF_TS                                # from Cell 2
where_sql = f"updated_at_ts='{filter_ts}' AND kg_content_type='data'"

JOIN_KEY       = "p_id"
CAST_TO_STRING = {"last_updated_attrs"}

qa_fqn   = f"iceberg_catalog1.{db}.{src_view}"
prod_fqn = f"iceberg_catalog2.{db}.{tgt_view}"

old_df = spark.table(qa_fqn).where(where_sql)
new_df = spark.table(prod_fqn).where(where_sql)
compare_cols = [c for c in old_df.columns if c != JOIN_KEY]

def _rename(prefix, df):
    return df.select(
        F.col(JOIN_KEY),
        *[
            (F.col(c).cast("string").alias(f"{prefix}__{c}")
             if c in CAST_TO_STRING
             else F.col(c).alias(f"{prefix}__{c}"))
            for c in compare_cols
        ],
    )

joined = _rename("old", old_df).join(_rename("new", new_df), on=JOIN_KEY, how="full_outer")

mismatch_conds = [~F.col(f"old__{c}").eqNullSafe(F.col(f"new__{c}")) for c in compare_cols]
any_mismatch = mismatch_conds[0]
for cond in mismatch_conds[1:]:
    any_mismatch = any_mismatch | cond

mismatched = joined.filter(any_mismatch).withColumn(
    "mismatched_columns",
    F.concat_ws(", ", *[
        F.when(~F.col(f"old__{c}").eqNullSafe(F.col(f"new__{c}")), F.lit(c))
        for c in compare_cols
    ]),
)
pair_cols = [col for c in compare_cols for col in (f"old__{c}", f"new__{c}")]
result = mismatched.select(JOIN_KEY, "mismatched_columns", *pair_cols)

print(f"Source : {qa_fqn}")
print(f"Target : {prod_fqn}")
print(f"Filter : {where_sql}")
print(f"QA   row count  : {old_df.count():,}")
print(f"PROD row count  : {new_df.count():,}")
print(f"Mismatched rows : {mismatched.count():,}")
result.show(truncate=False)

print("\n--- Mismatch count per column ---")
for c in compare_cols:
    cnt = joined.filter(~F.col(f"old__{c}").eqNullSafe(F.col(f"new__{c}"))).count()
    if cnt > 0:
        print(f"  {c}: {cnt:,}")

### Batch view migrate

**What it does.** Applies every spec in `view_specs` to prod via the
6-step DDL rewrite. Writes each outcome to `view_state.csv`.

**Run when.** You've piloted at least one view (Cell 18) and trust the
rewrite.

**Gotcha.** `skip_if_ok=True` (default) skips rows already
`migrate_status=success`. To force a re-migrate on a row, wipe its
status cell in JupyterLab's CSV viewer or pass `skip_if_ok=False`.

In [ ]:
state_df = migrate_views(spark, bundle, view_specs, skip_if_ok=True)

# Display the migrate-relevant columns only.
keep = ("view_suffix", "qa_view", "prod_view",
        "migrate_status", "migrate_at", "migrate_error")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120, "display.width", 240):
    display(state_df[[c for c in keep if c in state_df.columns]])

### Batch view validate

**What it does.** For every spec, counts both sides under the spec's
`filter_expression`, full-outer joins on `join_key`, computes total
mismatched rows. Verdict (`ok` / `mismatch` / `error`) lands in
`view_state.csv`.

**Run when.** Cell 22 has finished.

**Gotcha.** `per_column=True` is much slower (N extra COUNTs per view)
but prints which columns drift, inline. Per-column counts are NOT
persisted — only `mismatched_rows` (total) is stored.

In [ ]:
state_df = validate_views(spark, bundle, view_specs,
                          skip_if_ok=True, per_column=False)

# Hide migrate_* columns from the displayed slice for readability.
keep = ("view_suffix", "qa_view", "prod_view",
        "validation_status", "validation_at",
        "qa_count", "prod_count", "mismatched_rows", "validation_error")
# Cast count columns to nullable Int64 so big numbers don't go to scientific.
view = state_df[[c for c in keep if c in state_df.columns]].copy()
for c in ("qa_count", "prod_count", "mismatched_rows"):
    if c in view.columns:
        view[c] = pd.to_numeric(view[c], errors="coerce").astype("Int64")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120, "display.width", 240):
    display(view)

### Property sync — single-table inspect

**What it does.** For one QA table + its prod counterpart, prints
TBLPROPERTIES via the SQL view, via the Iceberg view, and a side-by-side
diff (`status ∈ {equal, value_changed, missing_on_prod, extra_on_prod}`).

**Run when.** Deciding which property keys are worth copying QA → prod.

**Gotcha.** Iceberg-managed reserved keys (`current-snapshot-id`,
`snapshot-count`, etc.) appear in SQL output but can't be SET via
`ALTER`. `table_properties.RESERVED_KEYS` enumerates them; the planner
in Cell 28 skips them automatically.

In [ ]:
SRC_CATALOG = "iceberg_catalog1"
TGT_CATALOG = "iceberg_catalog2"

TABLE_SUFFIX = "sds_em__assessment__publish"
QA_FQN   = f"{SRC_CATALOG}.kg_olap_v3.{TABLE_SUFFIX}"
PROD_FQN = f"{TGT_CATALOG}.kg_olap.{TABLE_SUFFIX}"

print(f"QA   : {QA_FQN}")
print(f"prod : {PROD_FQN}")

df_qa = compare_properties(spark, QA_FQN)
print(f"\nQA properties: {len(df_qa)} keys")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 200):
    display(df_qa)

df_prod = compare_properties(spark, PROD_FQN)
print(f"\nprod properties: {len(df_prod)} keys")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 200):
    display(df_prod)

df_diff = diff_qa_vs_prod(spark, QA_FQN, PROD_FQN)
print(f"\nstatus counts: {df_diff['status'].value_counts().to_dict()}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 200):
    display(df_diff)

### Property sync — plan

**What it does.** Builds (qa_fqn, prod_fqn) pairs for every prod schema
in `QA_SCHEMA_FOR`, then for `KEY` computes per-pair `action ∈ {skip
(qa missing), noop, set}`.

**Run when.** After Cell 26 has identified a key worth syncing.

**Gotcha.** Pairs are discovered **from the prod side** (`SHOW TABLES
IN iceberg_catalog2.<prod_schema>`) — only tables that already exist in
prod are paired. Uncomment additional `QA_SCHEMA_FOR` rows to cover
more batches in the same run.

In [ ]:
KEY = "write.parquet.page-size-bytes"

# Map prod schema -> QA schema (handles the QA->prod schema rename).
# Uncomment additional batches as needed.
QA_SCHEMA_FOR = {
    "kg_olap":    "kg_olap_v3",
    # "kg_publish": "kg_publish_final",
}
PROD_SCHEMAS = list(QA_SCHEMA_FOR.keys())

# Discover pairs from the PROD side — only sync properties for tables that
# actually exist in prod post-migration.
pairs: list[tuple[str, str]] = []
for prod_schema in PROD_SCHEMAS:
    qa_schema = QA_SCHEMA_FOR[prod_schema]
    for r in spark.sql(f"SHOW TABLES IN iceberg_catalog2.{prod_schema}").collect():
        t = r[1]
        pairs.append((
            f"iceberg_catalog1.{qa_schema}.{t}",
            f"iceberg_catalog2.{prod_schema}.{t}",
        ))
print(f"{len(pairs)} pair(s) across {len(PROD_SCHEMAS)} prod schema(s)")

# Per-pair plan for KEY only.
#   action = "skip (qa missing)"  -> nothing to copy from
#            "noop"                -> already in sync
#            "set"                 -> ALTER TABLE will run in the apply cell
rows = []
for qa_fqn, prod_fqn in pairs:
    qa_val   = properties_via_iceberg(spark, qa_fqn).get(KEY)
    prod_val = properties_via_iceberg(spark, prod_fqn).get(KEY)
    rows.append({
        "prod_fqn":   prod_fqn,
        "qa_value":   qa_val,
        "prod_value": prod_val,
        "action":     "skip (qa missing)" if qa_val is None
                      else "noop"          if qa_val == prod_val
                      else "set",
    })

df_plan = pd.DataFrame(rows)
print(f"\nKEY    : {KEY}")
print(f"action counts: {df_plan['action'].value_counts().to_dict()}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 200):
    display(df_plan)

### Property sync — apply (safety-gated)

**What it does.** Iterates `df_plan`, runs `ALTER TABLE SET
TBLPROPERTIES` for every row with `action='set'`. Captures per-row
`applied` / `error` and a timestamp.

**Run when.** You've reviewed `df_plan` from Cell 28.

**Gotcha.** `apply = False` default — no-op until you flip it. Single
quotes in `qa_value` are escaped inline before the SQL literal is
built; this guards against accidental SQL syntax breaks on property
values with apostrophes.

In [ ]:
from datetime import datetime, timezone

# ============================================================
#   STOP. Read df_plan above before flipping `apply = True`.
#   - Only rows with action='set' will be touched.
#   - ALTER TABLE SET TBLPROPERTIES is non-destructive but real.
# ============================================================
apply = False

if not apply:
    print("This cell is a no-op while apply=False. Set apply=True and re-run to execute.")
else:
    results = []
    for _, row in df_plan.iterrows():
        rec = {**row.to_dict(), "applied": False, "error": "",
               "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        if row["action"] == "set":
            try:
                # Escape single quotes in the value to keep the SQL literal valid.
                qa_v = str(row["qa_value"]).replace("'", "''")
                spark.sql(
                    f"ALTER TABLE {row['prod_fqn']} "
                    f"SET TBLPROPERTIES ('{KEY}'='{qa_v}')"
                )
                rec["applied"] = True
            except Exception as e:
                rec["error"] = str(e)
        results.append(rec)

    df_apply = pd.DataFrame(results)
    n_set    = int((df_apply["action"] == "set").sum())
    n_done   = int(df_apply["applied"].sum())
    n_errors = int((df_apply["error"] != "").sum())
    print(f"applied: {n_done} / {n_set}")
    print(f"errors : {n_errors}")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 200):
        display(df_apply[["prod_fqn", "qa_value", "prod_value", "action", "applied", "error"]])

### Property sync — verify

**What it does.** Re-reads `KEY` from every prod table in `pairs` after
apply.

**Run when.** Immediately after Cell 30 with `apply=True`.

**Gotcha.** Only checks `KEY`. If you are syncing multiple keys, run
this cell once per key (or extend it to iterate over a list of keys).

In [ ]:
# Re-read KEY on every prod table to confirm the sync took effect.
check = []
for _, prod_fqn in pairs:
    v = properties_via_iceberg(spark, prod_fqn).get(KEY)
    check.append({"prod_fqn": prod_fqn, KEY: v})

with pd.option_context("display.max_rows", None, "display.max_colwidth", 200):
    display(pd.DataFrame(check))

### Backup rename — config

**What it does.** Declares which prod schemas to back up (table-by-table
rename into `<name><SUFFIX>`) and the version suffix for this run.

**Run when.** You need to preserve the current prod state before a
destructive op — e.g. re-migrating from scratch over the existing
tables.

**Gotcha.** Bump `SUFFIX` per run (`_v1` → `_v2` → `_v3`) so old
backups aren't overwritten. The cell prints the new names so you can
verify before running the apply cell.

In [ ]:
# Backup-and-destroy: rename each prod schema in TARGETS to <name><SUFFIX>.
# Bump SUFFIX per run if you want multiple historical backups
# (e.g. "_v1" -> "_v2" -> "_v3" ...).
CATALOG = "iceberg_catalog2"
TARGETS = ["kg_publish", "kg_olap"]
SUFFIX  = "_v2"

new_names = [f"{t}{SUFFIX}" for t in TARGETS]

print(f"catalog  : {CATALOG}")
print(f"targets  : {TARGETS}")
print(f"suffix   : {SUFFIX}")
print(f"new names: {new_names}")

### Backup rename — apply (safety-gated)

**What it does.** For each target schema: lists tables + views, drops
the views, creates the new `<name><SUFFIX>` schema, renames every table
into it, drops the now-empty old schema.

**Run when.** You've reviewed Cell 34's printout and confirmed the
target schemas + suffix.

**Gotcha.** `confirm = False` default — flip to True to execute. Views
are DROPPED (not renamed) — re-create them via Cells 16/22 after the
rename if needed. Fails fast on the first error: a half-renamed schema
needs manual recovery.

In [ ]:
# ============================================================
#   STOP. Read the Cell 34 printout before flipping
#   `confirm = True`.
#   - ALTER TABLE ... RENAME TO moves every table from <OLD> to <OLD><SUFFIX>.
#   - DROP VIEW removes view DDLs (re-create later with Cells 16/22 if needed).
#   - DROP SCHEMA removes the empty original namespace.
#   - Fails fast: if one ALTER TABLE fails, the loop aborts mid-batch and the
#     schema ends up half-renamed. Recovery is manual.
# ============================================================
confirm = False

if not confirm:
    print("This cell is a no-op while confirm=False. Set confirm=True and re-run to rename.")
else:
    for OLD in TARGETS:
        NEW = f"{OLD}{SUFFIX}"
        print(f"\n=== {OLD} -> {NEW} ===")

        try:
            rels = [r[1] for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{OLD}").collect()]
        except Exception as e:
            print(f"  SKIP: {OLD} not present in {CATALOG} ({e})")
            continue

        try:
            vws = [r[1] for r in spark.sql(f"SHOW VIEWS IN {CATALOG}.{OLD}").collect()]
        except Exception:
            vws = []
        tbls = [r for r in rels if r not in vws]
        print(f"  found: {len(tbls)} table(s), {len(vws)} view(s)")

        for v in vws:
            spark.sql(f"DROP VIEW IF EXISTS {CATALOG}.{OLD}.{v}")
            print(f"  dropped view {v}")

        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{NEW}")
        print(f"  created schema {NEW}")

        for t in tbls:
            spark.sql(f"ALTER TABLE {CATALOG}.{OLD}.{t} RENAME TO {CATALOG}.{NEW}.{t}")
            print(f"  renamed {t}")

        spark.sql(f"DROP SCHEMA {CATALOG}.{OLD}")
        print(f"  dropped schema {OLD}")

    print("\nDONE. Run the verify cell to confirm.")

### Backup rename — verify

**What it does.** Lists namespaces in the target catalog and counts
tables in each new backup schema.

**Run when.** After Cell 36 with `confirm=True`.

**Gotcha.** Read-only.

In [ ]:
ns = [r[0] for r in spark.sql(f"SHOW NAMESPACES IN {CATALOG}").collect()]
print(f"Namespaces in {CATALOG}:")
for s in sorted(ns):
    print(f"  - {s}")

print()
for OLD in TARGETS:
    NEW = f"{OLD}{SUFFIX}"
    try:
        cnt = len(spark.sql(f"SHOW TABLES IN {CATALOG}.{NEW}").collect())
        print(f"{CATALOG}.{NEW}: {cnt} table(s)")
    except Exception as e:
        print(f"{CATALOG}.{NEW}: ERROR {e}")